# Natural Language Processing

Krish Ram  
CSCI 130-41083  
Assignment 4  
November 17, 2026

## 1.1 NLP pipeline

## Tokenization

Tokenization splits text into individual words and punctuation.

In [1]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def tokenize(text):
    return word_tokenize(text)

sample_text = 'The movie was surprisingly good!'
tokens = tokenize(sample_text)
print(tokens)

['The', 'movie', 'was', 'surprisingly', 'good', '!']


## Lowercasing

Lowercasing makes words consistent before comparing them.

In [2]:
def lowercase(tokens):
    lowercase_tokens = []
    for token in tokens:
        lowercase_tokens.append(token.lower())
    return lowercase_tokens

lowercase_tokens = lowercase(tokens)
print(lowercase_tokens)

['the', 'movie', 'was', 'surprisingly', 'good', '!']


## Punctuation removal

Remove punctuation from the tokens, keeping letters and numbers.

In [3]:
def remove_punctuation(tokens):
    cleaned_tokens = []
    for token in tokens:
        cleaned_token = ""
        for character in token:
            if character.isalnum():
                cleaned_token += character
        if cleaned_token:
            cleaned_tokens.append(cleaned_token)
    return cleaned_tokens

cleaned_tokens = remove_punctuation(lowercase_tokens)
print(cleaned_tokens)

['the', 'movie', 'was', 'surprisingly', 'good']


## Stopword removal

Stopwords are common words that can be removed when they are not needed for a task.

In [4]:
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    filtered_tokens = []
    for token in tokens:
        if token not in stop_words:
            filtered_tokens.append(token)
    return filtered_tokens

filtered_tokens = remove_stopwords(cleaned_tokens)
print(filtered_tokens)

['movie', 'surprisingly', 'good']


### Lemmatization

In [5]:
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet", quiet=True)
lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    lemmas = []
    for token in tokens:
        lemmas.append(lemmatizer.lemmatize(token))
    return lemmas

lemmas = lemmatize(filtered_tokens)
print(lemmas)

['movie', 'surprisingly', 'good']


## 1.2 Stopwords and meaning

removing a common word can change its meaning.

In [6]:
sentence = "The movie was not good and I would not recommend it."

sentence_tokens = tokenize(sentence)
sentence_tokens = lowercase(sentence_tokens)
sentence_tokens = remove_punctuation(sentence_tokens)
print("Before stopword removal:", sentence_tokens)

sentence_tokens = remove_stopwords(sentence_tokens)
print("After stopword removal:", sentence_tokens)

sentence_tokens = lemmatize(sentence_tokens)
print("After lemmatization:", sentence_tokens)

Before stopword removal: ['the', 'movie', 'was', 'not', 'good', 'and', 'i', 'would', 'not', 'recommend', 'it']
After stopword removal: ['movie', 'good', 'would', 'recommend']
After lemmatization: ['movie', 'good', 'would', 'recommend']


# To Remove or Not To Remove

stopwords are words that you remove before analyzing text. super common words that provide little information can be useful to remove. words like "the" and "is" are good examples. Removing them can cut down on processing and simplify inputs to just their most critical pieces. In this example of movie reviews, by removing "not" we limit the model's understanding of the actual meaning of the review. 

it's a bad idea to remove words that are critical to where and how the model will fulfill it's purpose, so we typically keep a stopword list to very common words that a model simply does not need to see to gauge meaning and intent

in this case, removing "not" hurt our ability to accurately understand the input. 

## 1.3 Stemming versus lemmatization

- **Stemming** uses rules to shorten words. The result may not be a complete word.
- **Lemmatization** uses vocabulary information to find a dictionary form. Its result depends on the part of speech supplied.
- For example, the word `studies` produces different results with these two methods.

In [7]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
word = "studies"

print("Word:", word)
print("Stem:", stemmer.stem(word))
print("Lemma:", lemmatizer.lemmatize(word))

Word: studies
Stem: studi
Lemma: study


## Part 2: Classical sentiment classifier

Use the pipeline above to prepare the movie reviews, then train and test a classifier.

In [8]:
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split

nltk.download("movie_reviews", quiet=True)

review_ids = movie_reviews.fileids()
review_labels = []
for review_id in review_ids:
    review_labels.append(movie_reviews.categories(review_id)[0])

train_ids, test_ids, train_labels, test_labels = train_test_split(
    review_ids, review_labels, test_size=0.2, random_state=42, stratify=review_labels
)
print("Training reviews:", len(train_ids))
print("Test reviews:", len(test_ids))

Training reviews: 1600
Test reviews: 400


In [9]:
def prepare_review(text):
    tokens = tokenize(text)
    tokens = lowercase(tokens)
    tokens = remove_punctuation(tokens)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    return " ".join(tokens)

train_texts = []
for review_id in train_ids:
    train_texts.append(prepare_review(movie_reviews.raw(review_id)))

test_texts = []
for review_id in test_ids:
    test_texts.append(prepare_review(movie_reviews.raw(review_id)))

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

vectorizer = CountVectorizer()
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors = vectorizer.transform(test_texts)

classifier = MultinomialNB()
classifier.fit(train_vectors, train_labels)
predictions = classifier.predict(test_vectors)

print("Accuracy:", accuracy_score(test_labels, predictions))
print("Confusion matrix (rows: actual; columns: predicted)")
print("Order: negative, positive")
print(confusion_matrix(test_labels, predictions, labels=["neg", "pos"]))

Accuracy: 0.8125
Confusion matrix (rows: actual; columns: predicted)
Order: negative, positive
[[163  37]
 [ 38 162]]


### Ten test reviews

Five correct and five incorrect predictions are selected from the test set. The original review text and labels are kept for the language model comparison.

In [11]:
correct_reviews = []
incorrect_reviews = []

for index in range(len(test_ids)):
    review = (test_ids[index], movie_reviews.raw(test_ids[index]),
              test_labels[index], predictions[index])
    if test_labels[index] == predictions[index] and len(correct_reviews) < 5:
        correct_reviews.append(review)
    elif test_labels[index] != predictions[index] and len(incorrect_reviews) < 5:
        incorrect_reviews.append(review)

selected_reviews = correct_reviews + incorrect_reviews
for review_id, text, actual, predicted in selected_reviews:
    print(review_id, " actual:", actual, "predicted:", predicted)

neg/cv748_14044.txt  actual: neg predicted: neg
neg/cv020_9234.txt  actual: neg predicted: neg
pos/cv088_24113.txt  actual: pos predicted: pos
neg/cv491_12992.txt  actual: neg predicted: neg
pos/cv521_15828.txt  actual: pos predicted: pos
neg/cv466_20092.txt  actual: neg predicted: pos
neg/cv106_18379.txt  actual: neg predicted: pos
pos/cv799_18543.txt  actual: pos predicted: neg
pos/cv028_26746.txt  actual: pos predicted: neg
pos/cv841_3967.txt  actual: pos predicted: neg


In [12]:
for review_id, text, actual, predicted in selected_reviews:
    print(f"\n--- {review_id} ---")
    print(text)


--- neg/cv748_14044.txt ---
dr dolittle ( 20th century fox ) running time : 1 hour 25 minutes starring eddie murphy directed by betty thomas riding high on the success of the nutty professor ( 1996 ) , murphy returns in this abysmal comedy . 
he plays doctor john dolittle , who as a child had the ability to understand animals . 
however , after being 'exorcised' he loses this ability , and we fast forward to see dolittle in a crummy job surrounding by crummy people ( most notably dr mark weller , played by oliver platt . ) 
however , with a bonk on the head his ability to understand animals returns , and 'comedy' is ready and waiting to happen . 
except comedy never does happen . 
doctor dolittle , even with the excellent premise , rarely manages to raise a laugh . 
poor old murphy looks bored stiff throughout the whole movie , and his performance suffers . 
the manic murphy we saw in the 80's is gone , his more mature ( and more boring ) twin appears to be getting all the work in the

## Classification by LLM:

Model was QWEN 3.8 Flash-Next served locally on my machine. the cost of 1000 classifications is only the cost of electricity. the model consistently rated 9/10 correctly. I also considered deploying a system-1 model for this assignment, because classification from free-form input is the domain of those models, and they are somewhat state of the art. Setting up the local LLM I tested took about four hours.

### Bare Prompt:  

Prompt: Is this movie review positive or negative?  (review text from cell above)

Response:
Dr. Dolittle — NEGATIVE
The reviewer is clearly disappointed with the film, criticizing its weak humor, uninspired performances, and unconvincing animal effects. Although a few jokes and characters receive praise, the overall assessment is overwhelmingly negative.

Spawn — NEGATIVE
The reviewer strongly dislikes the film, describing its story as underdeveloped and its special effects as unconvincing. Even the action sequences fail to impress. The conclusion makes the sentiment unmistakable, calling the movie outright awful.

Where the Money Is — POSITIVE
The reviewer appreciates the film's entertaining story, charismatic characters, and strong performances from Paul Newman and Linda Fiorentino. Particular praise goes to the humor and suspenseful heist sequence. The overall tone is enthusiastic and approving.

Dirty Work — POSITIVE
This review is interesting because the writer acknowledges numerous flaws, including crude humor, unsympathetic characters, and a weak plot. However, their personal enjoyment outweighs those criticisms. The reviewer gives it a high rating and emphasizes how much they laughed, making the overall sentiment positive.

It's a Wonderful Life — POSITIVE
The reviewer is exceptionally enthusiastic, describing the movie as a phenomenal classic. While acknowledging some dated elements and excessive sentimentality, they consider the performances, emotional impact, and message outstanding. This is an overwhelmingly positive review.

eXistenZ — NEGATIVE
The reviewer finds the movie disappointing compared with David Cronenberg's earlier work. Criticisms include predictable storytelling, unnecessary gore, and lackluster performances. Although the production design receives some praise, the overall judgment remains negative.

The Twilight Murders — NEGATIVE
The reviewer appreciates the film's exploration of social and political issues and acknowledges some competent directing and acting. However, they consider the story formulaic and unoriginal, with uneven performances. The criticisms outweigh the praise, resulting in a mildly negative assessment.

True Crime — POSITIVE
The reviewer praises Clint Eastwood's performance and direction, along with the film's character development and engaging dramatic moments. Although the ending is criticized for shifting toward conventional action, the reviewer considers the film an intriguing and worthwhile drama.

Book of Shadows: Blair Witch 2 — POSITIVE
This is a particularly mixed review. The writer considers the sequel inferior to the original, criticizing its complicated story and diminished originality. However, they also praise its cinematography, acting, and atmosphere. Most importantly, the reviewer explicitly says it is not a bad film, making the overall assessment cautiously positive.

Desperado — POSITIVE
The reviewer enjoys the film's exaggerated action, dark humor, and energetic direction. Although they acknowledge the lack of emotional depth and excessive violence, these elements do not diminish their enjoyment. The enthusiastic tone and description of the movie as entertaining fun establish a positive sentiment.


### Constrained Prompt:
Prompt: Classify this movie review as positive or negative. Answer with exactly one word: POSITIVE or NEGATIVE. (one segment each from the above cell)

Responses:

NEGATIVE

NEGATIVE

POSITIVE

POSITIVE

POSITIVE

NEGATIVE

NEGATIVE

POSITIVE

POSITIVE

POSITIVE


### Few Shot Prompt:

(All review text from above cell provided at once)

Prompt:Classify these movie reviews as POSITIVE or NEGATIVE.
Answer with exactly one word for each: POSITIVE or NEGATIVE.


Response (review IDs added afterward):

neg/cv748_14044.txt: NEGATIVE  
neg/cv020_9234.txt: NEGATIVE  
pos/cv088_24113.txt: POSITIVE  
neg/cv491_12992.txt: POSITIVE  
pos/cv521_15828.txt: POSITIVE  
neg/cv466_20092.txt: NEGATIVE  
neg/cv106_18379.txt: NEGATIVE  
pos/cv799_18543.txt: POSITIVE  
pos/cv028_26746.txt: POSITIVE  
pos/cv841_3967.txt: POSITIVE  

### Responses by review

| Review ID | Dataset label | Bare | Constrained | Third prompt |
| --- | --- | --- | --- | --- |
| neg/cv748_14044.txt | NEGATIVE | NEGATIVE | NEGATIVE | NEGATIVE |
| neg/cv020_9234.txt | NEGATIVE | NEGATIVE | NEGATIVE | NEGATIVE |
| pos/cv088_24113.txt | POSITIVE | POSITIVE | POSITIVE | POSITIVE |
| neg/cv491_12992.txt | NEGATIVE | POSITIVE | POSITIVE | POSITIVE |
| pos/cv521_15828.txt | POSITIVE | POSITIVE | POSITIVE | POSITIVE |
| neg/cv466_20092.txt | NEGATIVE | NEGATIVE | NEGATIVE | NEGATIVE |
| neg/cv106_18379.txt | NEGATIVE | NEGATIVE | NEGATIVE | NEGATIVE |
| pos/cv799_18543.txt | POSITIVE | POSITIVE | POSITIVE | POSITIVE |
| pos/cv028_26746.txt | POSITIVE | POSITIVE | POSITIVE | POSITIVE |
| pos/cv841_3967.txt | POSITIVE | POSITIVE | POSITIVE | POSITIVE |

Correct labels: bare 9/10; constrained 9/10; third prompt 9/10.

## Where they disagreed:

our classifier got 5/10 right. The LLM got 9/10. the LLM consistently missed the "Dirty Work" review but the classical classifier got correct based on the dataset label. 

Those ten were deliberately chosen such that 5 reviews the classical classifier got correct, and five that it missed to see if there was a difference. Of course, modern LLMs are stupid advanced and seemed to form a consistent impression of each review. The Weird one, "Dirty Work", was strange because the model explicitly reasoned about the reviewer's enjoyment and overall sentiment of the review to reach it's conclusion, and seemed to place weight on the reviewer's enjoyment and experience rather than the actual content of their review.


 our classifier was about 80% accurate, and the LLM showed 90% accuracy (on the ten samples). I think over 10 million reviews, the classifier would suffice and could be used, especially given that each review fed to the LLM would be an expense. in my case, the cost was up-front in hardware, but if I wanted to do these reviews to a frontier model like CHATGPT, i would need to pay an API fee at a rate of 2$ per million input tokens and 10$ per million output tokens. that must add up quite severely in the long run.


## Testing reviews outside the movie domain

These product and restaurant reviews test the classical classifier on text outside its movie-review training data.

In [13]:
domain_reviews = [
    ("The coffee was hot and the staff remembered my order. I would go back.", "pos"),
    ("My headphones stopped working after a week, and the left side kept crackling.", "neg"),
    ("The noodles were cold by the time they reached our table.", "neg"),
    ("This backpack fits my laptop and still feels comfortable on my walk to work.", "pos"),
    ("The new desk wobbles every time I type. Tightening the screws did not help.", "neg"),
]

In [14]:
correct = 0

for review, label in domain_reviews:
    prepared = prepare_review(review)
    review_vector = vectorizer.transform([prepared])
    prediction = classifier.predict(review_vector)[0]
    print("Review:", review)
    print("Actual:", label, "Predicted:", prediction)
    print()
    if prediction == label:
        correct += 1

print("Correct:", correct, "out of", len(domain_reviews))

Review: The coffee was hot and the staff remembered my order. I would go back.
Actual: pos Predicted: pos

Review: My headphones stopped working after a week, and the left side kept crackling.
Actual: neg Predicted: pos

Review: The noodles were cold by the time they reached our table.
Actual: neg Predicted: pos

Review: This backpack fits my laptop and still feels comfortable on my walk to work.
Actual: pos Predicted: neg

Review: The new desk wobbles every time I type. Tightening the screws did not help.
Actual: neg Predicted: neg

Correct: 2 out of 5


## Where they disagreed Part 2

on out of domain data, the classifier did poorly, getting only 2/5 right. This suggests a poor ability to generalize, which makes sense given the specialized training data. 

## Prompt results

All three recorded prompt runs got 9/10 on the selected reviews. The bare prompt gave explanations along with its labels. The constrained prompt followed the one-word instruction. The third prompt also gave one-word answers. The predicted labels were the same across all three runs.

The LLM got all five reviews that the classical classifier missed correct according to the dataset labels. It labeled Dirty Work positive, while the dataset label and the classical classifier labeled it negative. The review itself includes both criticism and enjoyment, which makes that disagreement interesting.

## Comparison

| Question | Classical pipeline | Local LLM |
| --- | --- | --- |
| Accuracy on the ten selected reviews | 5/10 | 9/10 for each recorded prompt |
| Labeled training data needed | 1,600 labeled movie reviews | No task-specific training for this test |
| Time to get working | About 2 hours | About 4 hours to set up; 25–30 minutes to run the recorded prompts |
| Cost per 1,000 classifications | Electricity; not measured | Electricity; not measured. The computer cost nearly $2,000. |
| Can a decision be explained? | Word weights can help show what influenced a prediction. | The bare prompt gave explanations, but those explanations do not prove exactly how the model reached its answer. |
| Text unlike the training data | 2/5 correct on the product and restaurant reviews. | Not tested on those reviews. |

The ten movie reviews were chosen to include five classical successes and five failures, so their 5/10 result is not its overall test accuracy. The five out-of-domain examples are also too few to establish general performance.

For a company handling ten million reviews a month, I would use the classical classifier because it was relatively accurate on the held-out movie reviews and avoids a per-review API charge. I would switch to the LLM if the classifier missed enough important reviews that better handling of context justified the extra hardware and operating costs. The 9/10 LLM result is encouraging, but I would want a broader test before assuming it will perform that well on every review.